# 09 — GNN vs. baseline on Clearwell

Converting Riverside District's edges to a line graph (`nx.line_graph`), training a hand-rolled graph-convolution model on a geographic train/test split, comparing it against the Module 6 baseline classifier on the same split, then applying the trained model to Old Town's real untagged edges as a qualitative demo (no ground truth there, by design).

### Build the line graph and feature table for Riverside District

`nx.line_graph(riverside)` turns every edge `(u, v, k)` into a node. Each line-graph node's features come straight from the original edge's attributes, plus `street_count` and `x` pulled from the edge's `u` endpoint (same pattern as the Module 6 baseline).

In [ ]:
import sys
sys.path.append("..")

import networkx as nx
import torch
import pandas as pd
from sklearn.metrics import classification_report

from src.data.synthetic_city import load_clearwell
from src.ml.features import build_feature_table

G = load_clearwell()
riverside_nodes = [n for n in G.nodes if n.startswith("rv_")]
riverside = G.subgraph(riverside_nodes).copy()

L = nx.line_graph(riverside)
lg_nodes = list(L.nodes())  # each is (u, v, k) from the original graph

edges_in_lg_order = [(u, v, k, riverside[u][v][k]) for (u, v, k) in lg_nodes]
df = build_feature_table(riverside, edges_in_lg_order, require_tagged=False)
df["lg_node"] = lg_nodes
df = df.set_index("lg_node")
print("rows:", len(df))
print(df["label"].value_counts())


### Build the feature matrix, adjacency matrix, and train/test masks

`nx.to_numpy_array(L, nodelist=lg_nodes)` turns the line graph into exactly the kind of adjacency matrix from the widget — same idea, just 98x98 instead of 4x4. Self-loops (`+ torch.eye(...)`) are added so a node's own features survive the averaging, not just its neighbors'. Instead of physically splitting the graph into two pieces (which would lose real cross-boundary connections), train/test are `mask`s over the *same* full graph — the GNN still sees the whole structure, but the loss/accuracy only ever look at one half's labels.

**Split on `y` (row), not `x` (column).** Riverside's label rule is defined by column (`c < 3` = accessible), so splitting on `x` would hand the model an almost all-one-class train set and an almost all-the-other-class test set — a real distribution-shift bug caught by both models scoring an identical, suspiciously bad 10% accuracy. Splitting on `y` instead keeps both classes well-represented on both sides, since the label rule never depends on row.

In [ ]:
import pandas as pd
from src.ml.gcn import normalized_adjacency

feature_cols = ["length", "street_count"]
categorical_cols = ["highway", "surface"]
encoded = pd.get_dummies(df[feature_cols + categorical_cols], columns=categorical_cols)

X = torch.tensor(encoded.values.astype(float), dtype=torch.float32)
y = torch.tensor(df["label"].values, dtype=torch.long)

A = nx.to_numpy_array(L, nodelist=lg_nodes)
A = torch.tensor(A, dtype=torch.float32)
A_norm = normalized_adjacency(A)

median_y = df["y"].median()
train_mask = torch.tensor((df["y"] < median_y).values)
test_mask = torch.tensor((df["y"] >= median_y).values)
print("train:", train_mask.sum().item(), " test:", test_mask.sum().item())


### The GCN model: two stacked graph-convolution layers

Same `SimpleGCNLayer` idea from the toy example (neighbor-average → `nn.Linear`), stacked twice: the first layer's output (with a `ReLU` in between) becomes the second layer's input, ending in 2 output scores (accessible / not) instead of raw features.

In [ ]:
from src.ml.gcn import SimpleGCN

torch.manual_seed(42)
model = SimpleGCN(in_features=X.shape[1], hidden_features=16, num_classes=2)
print(model)


### Train: this is where autograd actually does the work

`loss.backward()` is the one line that matters most here — PyTorch walks back through every operation that produced `loss` (the GCN layers, the matrix multiplications, everything) and computes how much each weight in `nn.Linear` contributed to the error. `optimizer.step()` then nudges every weight slightly in the direction that reduces the loss. 100 passes over the same (small) graph.

In [ ]:
from src.ml.gcn import train_gcn

model = train_gcn(model, X, A_norm, y, train_mask, epochs=100, lr=0.01, verbose=True)


### Evaluate: GNN vs. the Module 6 baseline, on the exact same split

`model.eval()` turns off training-only behavior (not used here, but it's the standard pattern). `torch.no_grad()` skips building the autograd machinery since we're not training anymore — just asking the already-trained model for predictions.

In [ ]:
from src.ml.baseline import train_baseline

model.eval()
with torch.no_grad():
    out = model(X, A_norm)
    preds = out.argmax(dim=1)

print("GNN classification report (test set):")
print(classification_report(y[test_mask].numpy(), preds[test_mask].numpy()))

rf = train_baseline(encoded[train_mask.numpy()], df["label"][train_mask.numpy()])
rf_preds = rf.predict(encoded[test_mask.numpy()])

print("\nBaseline (Random Forest) classification report (same test set):")
print(classification_report(df["label"][test_mask.numpy()], rf_preds))
